In [ ]:
import os;
from openai import OpenAI;
from IPython.display import display,Markdown;
import requests;
from bs4 import BeautifulSoup
import gradio as gr;

In [ ]:
ollama_url="http://localhost:11434/v1"
api_key="ollama"
MODEL = "llama3.2"
ollamaClient = OpenAI(
    base_url=ollama_url,
    api_key=api_key
)
system_prompt = """You are a professional business analyst. 
When given a company's website content, produce a clean research report with these sections:
- What they do (1-2 sentences)
- Who they serve (target customers)
- Key products or services
- Company tone / culture signals
- One thing that stands out
Keep it concise. Use bullet points."""


In [ ]:
def fetch_company_data(url):
    response = requests.get(url, headers={
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
    })

    if response.status_code != 200:
        return f"Failed to fetch {url} — status {response.status_code}"

    soup = BeautifulSoup(response.content, "html.parser")
    title = soup.title.string if soup.title else "No title"

    # Remove noise: scripts, styles AND structural chrome (nav, footer, etc.)
    for tag in soup(["script", "style", "img", "input", "nav", "footer", "header", "aside"]):
        tag.decompose()

    text = soup.body.get_text(separator="\n", strip=True) if soup.body else ""

    return (title + "\n\n" + text)[:5_000]


# YOUR TURN: change this to any company URL
company_url = "https://adobe.com"

website_text = fetch_company_data(company_url)

print(f"Fetched {len(website_text)} characters")
print("---")
print(website_text[:500])  # preview

In [ ]:

def call_llama(company_url):
    website_text = fetch_company_data(company_url)
    user_prompt = f"Here is the website content for {company_url}:\n\n{website_text}"
    response = ollamaClient.chat.completions.create(
        model=MODEL,
        messages=[
            {"role":"system","content":system_prompt},
            {"role":"user","content":user_prompt}
        ],
        stream=True
    )
    result=""
    for chunk in response:
         token = chunk.choices[0].delta.content or ""
         if token:
             result+=token
             yield result

In [ ]:
input = gr.Textbox(label="Enter your company url")
output = gr.Textbox(label="Summary",lines=1000)

gr.Interface(fn=call_llama,inputs=[input],outputs=[output]
             ,examples=[
                 "https://adobe.com"
             ]).launch();